# <center> <font size = 24 color = 'steelblue'>**Nova Haven - UrbanPulse Analytics**
**Complaint Classification and Routing Recommendations**

### Model 4: 311 Complaint Classification — NLP

- Multi-class text classification: predict the complaint type from text
- **Top 5 categories + "Other" (6 classes total):** Illegal Parking, HEAT/HOT WATER, Noise - Residential, Snow or Ice, Blocked Driveway, and Other (everything else)
- Use NLP techniques (TF-IDF + classifier, embeddings, or transformers)
- Handle real-world citizen text (informal language, abbreviations, multiple languages)
- **Minimum Benchmark:** Accuracy > 75%, weighted F1 > 0.70
- **Stretch Goal:** Accuracy > 85%, weighted F1 > 0.80
- **Also valued:** Complaint routing recommendations

In [2]:
import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# Model saving
import os
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True)
random.seed(42)
np.random.seed(42)

import torch
import langid
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm


In [3]:
#!pip install transformers sentencepiece torch langid

In [4]:
PROJECT_ROOT = Path.cwd().parent
#DATA_PATH = PROJECT_ROOT / "data" / "raw" / "urbanpulse_311_complaints.csv"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "urbanpulse_311_complaints_bilingual.csv"
OUTPUT_DIR = PROJECT_ROOT / "models" / "model4_nlp_classification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)

Project root: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting
Data path: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\data\raw\urbanpulse_311_complaints_bilingual.csv
Output dir: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\models\model4_nlp_classification


In [5]:
df = pd.read_csv(DATA_PATH)
print("Data shape:", df.shape)
df.head(3)

Data shape: (434722, 11)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,resolution_description,borough,open_data_channel_type,status
0,67874778,2026-02-06T14:57:28.000,2026-02-07T21:24:52.000,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,This complaint is a duplicate of a building-wi...,MANHATTAN,ONLINE,Closed
1,68207007,2026-03-04T10:40:04.000,2026-03-06T16:05:24.000,HPD,Department of Housing Preservation and Develop...,PLUMBING,WATER SUPPLY,HPD conducted an inspection of this complaint....,BRONX,ONLINE,Closed
2,68298021,2026-03-12T11:26:42.000,2026-03-12T12:34:47.000,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,The New York City Police Department responded ...,BROOKLYN,MOBILE,Closed


In [6]:
# Lowercase text columns only
df = df.apply(lambda col: col.str.lower() if col.dtype == "object" else col)

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-/&]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
# save original complaint_text and complaint_type (in original language)
df["complaint_text_orig"] = (
    df["descriptor"].fillna("").apply(clean_text) + " | " +
    df["resolution_description"].fillna("").apply(clean_text)
)

## Multilingual preprocessing (before setting `X` and `y`)

This step detects Spanish, Chinese, and Russian text in the raw input columns and translates only those non-English values to English using **offline Hugging Face models**. English values are left unchanged.


In [8]:
# =========================================================
# Multilingual handling BEFORE category mapping and X / y
# =========================================================
# This cell:
# 1) keeps original raw text for audit
# 2) detects row language from combined text
# 3) translates non-English text to English using offline HF models
# 4) then lowercases / cleans text for the rest of the notebook

# If needed:
# !pip install transformers sentencepiece torch langid tqdm

import re
import torch
import langid
from tqdm.auto import tqdm
from transformers import MarianMTModel, MarianTokenizer

TEXT_COLS = ["complaint_type", "descriptor", "resolution_description"]
SHORT_TEXT_COLS = ["complaint_type", "descriptor"]
LONG_TEXT_COLS = ["resolution_description"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -----------------------------
# Preserve originals for review
# -----------------------------
for col in TEXT_COLS:
    df[f"{col}_original"] = df[col]

# -----------------------------
# Helper functions
# -----------------------------
def safe_str(x):
    return "" if pd.isna(x) else str(x).strip()

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-/&]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_lang_detection_text(row):
    parts = [
        safe_str(row.get("complaint_type", "")),
        safe_str(row.get("descriptor", "")),
        safe_str(row.get("resolution_description", ""))
    ]
    txt = " | ".join([p for p in parts if p])
    return txt.strip()

def detect_lang_row(text):
    """
    Detect only the languages we care about for translation.
    Returns one of: en, es, zh, ru
    Falls back to 'en' for unknown/other languages.
    """
    text = safe_str(text)
    if text == "":
        return "en"

    lang, score = langid.classify(text)

    # normalize Chinese family codes if they appear
    if lang in ["zh", "zh-cn", "zh-tw"]:
        return "zh"
    if lang in ["es", "ru", "en"]:
        return lang

    # everything else treated as English for this notebook
    return "en"

MODEL_MAP = {
    "es": "Helsinki-NLP/opus-mt-es-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
    "ru": "Helsinki-NLP/opus-mt-ru-en",
}

_model_cache = {}

def get_model(lang):
    if lang not in _model_cache:
        model_name = MODEL_MAP[lang]
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name).to(device)
        model.eval()
        _model_cache[lang] = (tokenizer, model)
    return _model_cache[lang]

def translate_unique_values(values, lang, batch_size=32, max_length=128):
    """
    Translate a list-like of strings for one source language -> English.
    Returns dict: original_text -> translated_text
    """
    if lang == "en":
        return {}

    tokenizer, model = get_model(lang)

    vals = pd.Series(list(values)).dropna().astype(str).str.strip()
    vals = vals[vals != ""].unique().tolist()

    if len(vals) == 0:
        return {}

    translation_map = {}

    for i in tqdm(range(0, len(vals), batch_size), desc=f"Translating {lang}"):
        batch = vals[i:i + batch_size]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            generated = model.generate(**encoded)

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for src, tgt in zip(batch, decoded):
            translation_map[src] = tgt

    return translation_map

# -----------------------------
# Step 1: detect row language
# -----------------------------
df["lang_detection_text"] = df.apply(build_lang_detection_text, axis=1)
df["detected_language"] = df["lang_detection_text"].apply(detect_lang_row)

print("Detected language distribution:")
print(df["detected_language"].value_counts(dropna=False))

# -----------------------------
# Step 2: translate by ROW language
# This fixes short terms like 'carretera'
# -----------------------------
for lang in ["es", "zh", "ru"]:
    idx = df.index[df["detected_language"] == lang]
    if len(idx) == 0:
        continue

    print(f"\nProcessing language: {lang} | rows: {len(idx)}")

    # translate short categorical fields by row language
    for col in SHORT_TEXT_COLS:
        unique_vals = (
            df.loc[idx, col]
            .dropna()
            .astype(str)
            .str.strip()
        )
        unique_vals = unique_vals[unique_vals != ""].unique().tolist()

        trans_map = translate_unique_values(
            unique_vals,
            lang=lang,
            batch_size=32,
            max_length=64
        )

        df.loc[idx, col] = df.loc[idx, col].apply(
            lambda x: trans_map.get(str(x).strip(), x)
            if pd.notna(x) and str(x).strip() != ""
            else x
        )

    # translate longer narrative field by row language
    for col in LONG_TEXT_COLS:
        unique_vals = (
            df.loc[idx, col]
            .dropna()
            .astype(str)
            .str.strip()
        )
        unique_vals = unique_vals[unique_vals != ""].unique().tolist()

        trans_map = translate_unique_values(
            unique_vals,
            lang=lang,
            batch_size=16,
            max_length=256
        )

        df.loc[idx, col] = df.loc[idx, col].apply(
            lambda x: trans_map.get(str(x).strip(), x)
            if pd.notna(x) and str(x).strip() != ""
            else x
        )

# -----------------------------
# Optional override dictionary
# Good safety net for short descriptors
# -----------------------------
descriptor_override = {
    "carretera": "road",
    "calle": "street",
    "avenida": "avenue",
    "autopista": "highway"
}

df["descriptor"] = df["descriptor"].replace(descriptor_override)

# -----------------------------
# Lowercase text columns only
# after translation is complete
# -----------------------------
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)

def get_top_complaint_types(df: pd.DataFrame, n: int = 5) -> list:
    return df["complaint_type"].value_counts().head(n).index.tolist()

TOP_5 = get_top_complaint_types(df, n=5)
print("\nTop 5 complaint types after translation:")
print(TOP_5)

# quick QA sample
qa_cols = [
    "detected_language",
    "complaint_type_original", "complaint_type",
    "descriptor_original", "descriptor",
    "resolution_description_original", "resolution_description"
]

print("\nSample translated rows:")
display(df.loc[df["detected_language"] != "en", qa_cols].head(10))

Using device: cpu
Detected language distribution:
detected_language
en    424722
es     10000
Name: count, dtype: int64

Processing language: es | rows: 10000


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translating es:   0%|          | 0/4 [00:00<?, ?it/s]

Translating es:   0%|          | 0/12 [00:00<?, ?it/s]

Translating es:   0%|          | 0/17 [00:00<?, ?it/s]


Top 5 complaint types after translation:
['illegal parking', 'heat/hot water', 'noise - residential', 'snow or ice', 'blocked driveway']

Sample translated rows:


,detected_language,complaint_type_original,complaint_type,descriptor_original,descriptor,resolution_description_original,resolution_description
30,es,ruido - residencial,noise - residential,golpes/montajes,blows/assemblies,el departamento de policía de la ciudad de nue...,the police department of the city of new york ...
81,es,condición de la calle,condition of the street,bache,bump,el departamento de transporte inspeccionó esta...,the transport department inspected this compla...
85,es,mantenimiento o instalación,maintenance or installation,avistamiento de roedores,sighting of rodents,nyc parks ha completado la orden de trabajo so...,nyc parks has completed the requested work ord...
100,es,ruido - calle/sidewalk,noise - street/sidewalk,música fuerte/fiesta,loud music/party,el departamento de policía de la ciudad de nue...,the city police department of new york respond...
102,es,estacionamiento ilegal,illegal parking,hidratante bloqueado,blocked moisturizer,el departamento de policía de la ciudad de nue...,new york city police department responded to t...
123,es,plomo,lead,solicitud de equipo de plomo (residencial) (l10),request for lead equipment (residential) (l10),el departamento de protección ambiental (dep) ...,the environmental protection department (dep) ...
237,es,calidad del aire interior,indoor air quality,odor de aguas residuales,waste water odor,el departamento de salud de nueva york (dohmh)...,the health department of new york (dohmh) foun...
302,es,generalidades,general,cabinet,cabinet,hpd inspeccionó esta condición por lo que la q...,hpd inspected this condition so the complaint ...
380,es,estacionamiento ilegal,illegal parking,callejuelas bloqueadas,blocked alleys,el departamento de policía de la ciudad de nue...,new york city police department responded to t...
416,es,ruido,noise,ruido: construcción antes/después de las horas...,noise: construction before/after hours (nm1),el departamento de protección ambiental (dep) ...,the environmental protection department (dep) ...


In [9]:
print(df.iloc[215919])

unique_key                                                                  67912567
created_date                                                 2026-02-09t21:47:40.000
closed_date                                                  2026-02-10t03:05:16.000
agency                                                                          dsny
agency_name                                                 department of sanitation
complaint_type                                                           snow or ice
descriptor                                                                      road
resolution_description             the sanitation department investigated this co...
borough                                                                       queens
open_data_channel_type                                                        online
status                                                                        closed
complaint_text_orig                carretera | el departamento de

In [10]:


def get_top_complaint_types(df: pd.DataFrame, n: int = 5) -> list:
    return df["complaint_type"].value_counts().head(n).index.tolist()

TOP_5 = get_top_complaint_types(df, n=5)
TOP_5

['illegal parking',
 'heat/hot water',
 'noise - residential',
 'snow or ice',
 'blocked driveway']

In [11]:
def create_complaint_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map complaint types to the top 5 categories + "Other" (6 classes total).

    The top 5 categories are:
    - Illegal Parking
    - HEAT/HOT WATER
    - Noise - Residential
    - Snow or Ice
    - Blocked Driveway

    Everything else maps to "Other". This gives you 6 classes total —
    a much more manageable classification problem than 186 categories.
    """
    top_5 = ['illegal parking', 'heat/hot water', 'noise - residential',
             'snow or ice', 'blocked driveway']
    df['complaint_category'] = df['complaint_type'].apply(
        lambda x: x if x in top_5 else 'other'
    )

    print("Complaint category distribution:")
    print(df['complaint_category'].value_counts())

    coverage = df[df['complaint_category'] != 'other'].shape[0] / len(df) * 100
    print(f"\nTop 5 categories cover {coverage:.1f}% of all complaints")
    print(f"Total classes: {df['complaint_category'].nunique()} (top 5 + other)")

    return df

In [12]:
df_clean = create_complaint_categories(df)

Complaint category distribution:
complaint_category
other                  214963
illegal parking         66159
heat/hot water          62861
noise - residential     38931
snow or ice             27453
blocked driveway        24355
Name: count, dtype: int64

Top 5 categories cover 50.6% of all complaints
Total classes: 6 (top 5 + other)


In [13]:
df = df_clean

# Use richer text than descriptor alone
df["complaint_text"] = (
    df["descriptor"].fillna("").apply(clean_text) + " | " +
    df["resolution_description"].fillna("").apply(clean_text)
)

df["categories"] = df["complaint_category"].fillna("").apply(clean_text)

df = df[(df["complaint_text"] != "") & (df["categories"] != "")]
df = df.dropna(subset=["complaint_text", "categories"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["categories"].value_counts())

Dataset shape: (434722, 20)

Class distribution:
categories
other                  214963
illegal parking         66159
heat/hot water          62861
noise - residential     38931
snow or ice             27453
blocked driveway        24355
Name: count, dtype: int64


In [14]:
df.columns

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name',
       'complaint_type', 'descriptor', 'resolution_description', 'borough',
       'open_data_channel_type', 'status', 'complaint_text_orig',
       'complaint_type_original', 'descriptor_original',
       'resolution_description_original', 'lang_detection_text',
       'detected_language', 'complaint_category', 'complaint_text',
       'categories'],
      dtype='object')

In [15]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["categories"])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("Label mapping:")
print(id2label)

Label mapping:
{0: 'blocked driveway', 1: 'heat/hot water', 2: 'illegal parking', 3: 'noise - residential', 4: 'other', 5: 'snow or ice'}


In [16]:
### write this cleaned dataframe to data/processed/ folder for innovation - optimizer project

df.to_csv('../data/processed/urbanpulse_311_complaints_cleaned.csv', index=False)
#df.to_csv('/content/drive/MyDrive/cleaned_data/predicted_311_complaint_data.csv')
print("✓ Cleaned 311 Complaint data saved to ../data/processed/urbanpulse_311_complaints_cleaned.csv")

✓ Cleaned 311 Complaint data saved to ../data/processed/urbanpulse_311_complaints_cleaned.csv


In [49]:
### adding unique_key to X so that we can validate the predictions to the original data
X = df["complaint_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (347777,) (347777,)
Test:  (86945,) (86945,)


## Build the multilingual-friendly fast classifier

### improvement from the original notebook
Instead of English word n-grams, use:

- `analyzer="char_wb"`
- `ngram_range=(3, 5)`

This is usually much more tolerant of:
- spelling variation
- abbreviations
- transliteration
- multilingual text fragments

In [50]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        analyzer="char_wb",
        ngram_range=(3, 4),
        min_df=5,
        max_df=0.95,
        max_features=20000,
        sublinear_tf=True
    )),
    ("logreg", OneVsRestClassifier(
        LogisticRegression(
            max_iter=300,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        )
    ))
])

clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [51]:
# -----------------------------
# Evaluate
# -----------------------------
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy: 0.9654839266202772

Classification Report:
                     precision    recall  f1-score   support

   blocked driveway       1.00      1.00      1.00      4871
     heat/hot water       1.00      1.00      1.00     12572
    illegal parking       1.00      1.00      1.00     13232
noise - residential       0.73      1.00      0.84      7786
              other       1.00      0.93      0.96     42993
        snow or ice       1.00      1.00      1.00      5491

           accuracy                           0.97     86945
          macro avg       0.95      0.99      0.97     86945
       weighted avg       0.97      0.97      0.97     86945



In [52]:
### print performance metrics for this TF-IDF + Logistic Regression model:

y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

print (f"'Train Accuracy': {accuracy_score(y_train, y_train_pred):.4f}")
print (f"'Test Accuracy': {accuracy_score(y_test, y_test_pred):.4f}")
print (f"'Precision (weighted)': {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
print (f"'Recall (weighted)': {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
print (f"'F1 (weighted)': {f1_score(y_test, y_test_pred, average='weighted'):.4f}")

'Train Accuracy': 0.9663
'Test Accuracy': 0.9655
'Precision (weighted)': 0.9748
'Recall (weighted)': 0.9655
'F1 (weighted)': 0.9676


# Build Complaint Routing Recommendations

In [60]:
print(df.groupby(['categories', 'agency'])['agency_name'].count())

categories           agency
blocked driveway     nypd      24355
heat/hot water       hpd       62861
illegal parking      nypd      66159
noise - residential  nypd      38931
other                dcwp       1915
                     dep       20719
                     dhs        2279
                     dob       11126
                     doe         328
                     dohmh      6455
                     dot       38642
                     dpr        7178
                     dsny      24029
                     hpd       64987
                     nypd      34053
                     oos           7
                     oti          17
                     tlc        3228
snow or ice          dsny      27453
Name: agency_name, dtype: int64


Five of the 6 categories already have a default route-to-agency:

**"blocked driveway"** --- nypd

**"heat/hot water"** --- hpd

**"illegal parking"** --- nypd

**"noise - residential"** --- nypd

**"snow or ice"** --- dsny

However, we need to be able to route the complaints in the **"other"** category to the correct responding agency.

Below we do a quick EDA on the complaints that were categorized" as **"other"**

In [61]:
df_other = df[df['label']==4]
df_other.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,resolution_description,borough,open_data_channel_type,status,complaint_text_orig,complaint_type_orig,complaint_type_original,descriptor_original,resolution_description_original,lang_detection_text,detected_language,complaint_category,complaint_text,categories,label
1,68207007,2026-03-04t10:40:04.000,2026-03-06t16:05:24.000,hpd,department of housing preservation and develop...,plumbing,water supply,hpd conducted an inspection of this complaint....,bronx,online,closed,water supply | hpd conducted an inspection of ...,plumbing,plumbing,water supply,hpd conducted an inspection of this complaint....,plumbing | water supply | hpd conducted an ins...,en,other,water supply | hpd conducted an inspection of ...,other,4
6,67996182,2026-02-14t04:34:09.000,2026-02-14t07:30:31.000,nypd,new york city police department,noise - street/sidewalk,loud music/party,the new york city police department responded ...,brooklyn,mobile,closed,loud music/party | the new york city police de...,noise - street/sidewalk,noise - street/sidewalk,loud music/party,the new york city police department responded ...,noise - street/sidewalk | loud music/party | t...,en,other,loud music/party | the new york city police de...,other,4
9,67918125,2026-02-09t11:41:22.000,NaN,dohmh,department of health and mental hygiene,non-residential heat,inadequate or no heat,the department of health and mental hygiene ha...,bronx,online,in progress,inadequate or no heat | the department of heal...,non-residential heat,non-residential heat,inadequate or no heat,the department of health and mental hygiene ha...,non-residential heat | inadequate or no heat |...,en,other,inadequate or no heat | the department of heal...,other,4
10,68254352,2026-03-08t19:18:00.000,2026-03-09t12:22:00.000,dep,department of environmental protection,lead,lead kit request (residential) (l10),the department of environmental protection (de...,brooklyn,online,closed,lead kit request residential l10 | the departm...,lead,lead,lead kit request (residential) (l10),the department of environmental protection (de...,lead | lead kit request (residential) (l10) | ...,en,other,lead kit request residential l10 | the departm...,other,4
14,67928066,2026-02-10t11:11:28.000,2026-03-11t14:35:11.000,hpd,department of housing preservation and develop...,door/window,door,hpd conducted an inspection of this complaint....,brooklyn,phone,closed,door | hpd conducted an inspection of this com...,door/window,door/window,door,hpd conducted an inspection of this complaint....,door/window | door | hpd conducted an inspecti...,en,other,door | hpd conducted an inspection of this com...,other,4


In [62]:
# -----------------------------
# Encode labels for route-to-agency
# -----------------------------
label_encoder_rt = LabelEncoder()
df["rt_label"] = label_encoder_rt.fit_transform(df["agency"])

id2label_rt = {i: label for i, label in enumerate(label_encoder_rt.classes_)}
label2id_rt = {label: i for i, label in id2label.items()}

print("\nLabel mapping:")
print(id2label_rt)


Label mapping:
{0: 'dcwp', 1: 'dep', 2: 'dhs', 3: 'dob', 4: 'doe', 5: 'dohmh', 6: 'dot', 7: 'dpr', 8: 'dsny', 9: 'hpd', 10: 'nypd', 11: 'oos', 12: 'oti', 13: 'tlc'}


In [63]:
## use complaint_type to predict route_to_agency for "other" category

X_train_rt, X_test_rt, y_train_rt, y_test_rt = train_test_split(
    df["complaint_type"],
    df["rt_label"],
    test_size=0.2,
    random_state=42,
    stratify=df["rt_label"]
)

In [64]:

# -----------------------------
# Build Routing Recommendation pipeline
# -----------------------------
clf_rt = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight = 'balanced',
        n_jobs=-1
    ))
])

In [65]:

# -----------------------------
# Train
# -----------------------------
clf_rt.fit(X_train_rt, y_train_rt)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred_rt = clf_rt.predict(X_test_rt)

print("Accuracy:", accuracy_score(y_test_rt, y_pred_rt))
print("\nClassification Report:")
print(classification_report(y_test_rt, y_pred_rt, target_names=label_encoder_rt.classes_))

Accuracy: 0.9940537121168554

Classification Report:
              precision    recall  f1-score   support

        dcwp       1.00      1.00      1.00       383
         dep       1.00      1.00      1.00      4144
         dhs       0.58      1.00      0.74       456
         dob       0.98      0.96      0.97      2225
         doe       1.00      1.00      1.00        66
       dohmh       0.99      1.00      0.99      1291
         dot       1.00      1.00      1.00      7728
         dpr       1.00      1.00      1.00      1436
        dsny       1.00      1.00      1.00     10296
         hpd       1.00      1.00      1.00     25570
        nypd       1.00      0.99      0.99     32700
         oos       1.00      1.00      1.00         1
         oti       1.00      1.00      1.00         3
         tlc       1.00      1.00      1.00       646

    accuracy                           0.99     86945
   macro avg       0.97      1.00      0.98     86945
weighted avg       1.00    

In [66]:
y_train_pred_rt = clf_rt.predict(X_train_rt)
y_test_pred_rt = clf_rt.predict(X_test_rt)

In [67]:
### print performance metrics for this routing recommendation engine:

print (f"'Train Accuracy': {accuracy_score(y_train_rt, y_train_pred_rt):.4f}")
print (f"'Test Accuracy': {accuracy_score(y_test_rt, y_test_pred_rt):.4f}")
print (f"'Precision (weighted)': {precision_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")
print (f"'Recall (weighted)': {recall_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")
print (f"'F1 (weighted)': {f1_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")

'Train Accuracy': 0.9939
'Test Accuracy': 0.9941
'Precision (weighted)': 0.9956
'Recall (weighted)': 0.9941
'F1 (weighted)': 0.9945


In [68]:
### map recommended labels to agency and agency name

df.loc[X_train_rt.index, 'routing_label'] = y_train_pred_rt
df.loc[X_test_rt.index, 'routing_label'] = y_test_pred_rt

In [69]:
df['route_to_agency'] = df['routing_label'].map(id2label_rt)


In [70]:
agency_map = df[['agency', 'agency_name']].drop_duplicates().set_index('agency')['agency_name']
df['route_to agency_name'] = df['route_to_agency'].map(agency_map)

In [71]:
df[['categories', 'routing_label', 'route_to_agency', 'route_to agency_name']].value_counts()

categories           routing_label  route_to_agency  route_to agency_name                              
illegal parking      10.0           nypd             new york city police department                       66159
other                9.0            hpd              department of housing preservation and development    65253
heat/hot water       9.0            hpd              department of housing preservation and development    62861
noise - residential  10.0           nypd             new york city police department                       38931
other                6.0            dot              department of transportation                          38652
                     10.0           nypd             new york city police department                       32295
snow or ice          8.0            dsny             department of sanitation                              27453
blocked driveway     10.0           nypd             new york city police department                     

In [72]:
# Save NLP processed 311 Complaint data for visualization
df.to_csv('../data/processed/predicted_311_complaint_data.csv', index=False)
#df.to_csv('/content/drive/MyDrive/cleaned_data/predicted_311_complaint_data.csv')
print("✓ 311 Complaint data with routing recommendations saved to ../data/processed/predicted_311_complaint_data.csv")

✓ 311 Complaint data with routing recommendations saved to ../data/processed/predicted_311_complaint_data.csv


In [73]:
## Save models and artifacts

model_path = "../models/model4_nlp_classification/"


joblib.dump(clf, model_path + "model4_complaint_classifier_char_tfidf_logreg.pkl")
joblib.dump(clf_rt, model_path + "model4_routing_classifier_char_tfidf_logreg.pkl")
joblib.dump(label_encoder, model_path + "model4_category_label_encoder.pkl")
joblib.dump(label_encoder_rt, model_path + "model4_routing_label_encoder.pkl")

print("Saved artifacts to:", model_path)

Saved artifacts to: ../models/model4_nlp_classification/
